# Home Credit: offline relational LightGBM

Attach the official competition and the code-wheel dataset. This entry point builds applicant-level features, trains the full fixed-seed stratified five-fold recipe and saves predictions plus portable run artifacts. It does not contact private services.

Run from a fresh kernel. Code assets are checksum-verified before installation.

In [ ]:
import hashlib
import json
import subprocess
import sys
from pathlib import Path
from importlib.metadata import version

print({"python": sys.version, **{name: version(name) for name in ["numpy", "pandas", "lightgbm", "scikit-learn", "pyarrow"]}})

wheels = list(Path("/kaggle/input").rglob("home_credit_risk-*.whl"))
assert len(wheels) == 1, "Attach exactly one project wheel"
asset = json.loads((wheels[0].parent / "source.json").read_text())
assert asset["wheel"] == wheels[0].name
assert hashlib.sha256(wheels[0].read_bytes()).hexdigest() == asset["sha256"]
subprocess.run([sys.executable, "-m", "pip", "install", "--force-reinstall", "--no-index", "--no-deps", str(wheels[0])], check=True)

In [ ]:
import json
import os
import shutil
from home_credit.pipeline import prepare, fit_cv, sha256, write_json, source_identity, analyze

assert source_identity()["source_files"] == asset["source"]["source_files"]

roots = [Path("/kaggle/input/home-credit-default-risk"),
         Path("/kaggle/input/competitions/home-credit-default-risk")]
data_dir = next(p for p in roots if (p / "application_train.csv").is_file())
config_files = list(wheels[0].parent.glob("baseline.json"))
assert len(config_files) == 1
assert sha256(config_files[0]) == asset["configuration_sha256"]["baseline.json"]
config = json.loads(config_files[0].read_text())
config["threads"] = min(config["threads"], os.cpu_count() or 1)
cache = Path("/kaggle/working/features")
run_dir = Path("/kaggle/working/benchmark")
prepare(data_dir, cache)
manifest = fit_cv(cache, run_dir, config)
manifest["execution"] = {"platform": "kaggle", "wheel": wheels[0].name,
                         "wheel_sha256": sha256(wheels[0]),
                         "code_asset": json.loads((wheels[0].parent / "source.json").read_text())}
write_json(run_dir / "manifest.json", manifest)
analyze(cache, run_dir, Path("/kaggle/working/cv-report.json"))
shutil.copy2(run_dir / "submission.csv", "/kaggle/working/submission.csv")
shutil.make_archive("/kaggle/working/run-bundle", "zip", run_dir)
print(json.dumps(manifest["metrics"], indent=2))